# Python Notebook

This is a simple Python notebook for experimentation and testing. You can add your code, analysis, and visualizations here.

## GitHub PR Library Update Analyzer

This notebook analyzes GitHub Pull Requests to extract library version updates from POM file changes using local Ollama LLM.

### Prerequisites

1. **GitHub Personal Access Token**: 
   - Go to [GitHub Settings > Developer Settings > Personal Access Tokens > Tokens (classic)](https://github.com/settings/tokens)
   - Click "Generate new token (classic)"
   - Give it a descriptive name (e.g., "PR Analysis Tool")
   - Select the following scopes:
     - `repo` (Full control of private repositories) - needed to read PR data
     - Or just `public_repo` if you only need to access public repositories
   - Click "Generate token"
   - **Copy the token immediately** (you won't be able to see it again)
   - Store it securely

2. **Local Ollama**: 
   - Ensure Ollama is installed and running locally
   - Install a model (e.g., `ollama pull llama3` or `ollama pull mistral`)
   - Verify it's running: `ollama list`

### Step 1: Configure GitHub Access and PR Details

**Setting up your GitHub Token (Recommended):**

Instead of hardcoding your token in the notebook, store it in a `.env` file:

1. Create a `.env` file in the project root:
   ```bash
   echo "GITHUB_TOKEN=your_github_token_here" > .env
   ```

2. The `.env` file is already in `.gitignore` and won't be committed to version control.

3. The notebook will automatically load the token from the `.env` file using `python-dotenv`.

**Note:** The `.env` file should be in the same directory as this notebook.

In [15]:
# Define the PR to analyze using the full GitHub URL
PR_URL = "https://github.com/eg-internal/brand-to-eg-profile-sync/pull/596"

In [16]:
import os
import re
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Load GitHub token from environment variable
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise ValueError(
        "GITHUB_TOKEN environment variable not found!\n"
        "Please create a .env file in the project root with:\n"
        "  GITHUB_TOKEN=your_token_here\n"
    )

# Parse the PR URL to extract owner, repo, and PR number
pr_pattern = r'https://github\.com/([^/]+)/([^/]+)/pull/(\d+)'
match = re.match(pr_pattern, PR_URL)

if not match:
    raise ValueError(f"Invalid GitHub PR URL: {PR_URL}")

REPO_OWNER = match.group(1)
REPO_NAME = match.group(2)
PR_NUMBER = int(match.group(3))

print(f"✓ GitHub token loaded from .env file")
print(f"✓ Parsed PR URL: {PR_URL}")
print(f"  - Owner: {REPO_OWNER}")
print(f"  - Repository: {REPO_NAME}")
print(f"  - PR Number: {PR_NUMBER}")

✓ GitHub token loaded from .env file
✓ Parsed PR URL: https://github.com/eg-internal/brand-to-eg-profile-sync/pull/596
  - Owner: eg-internal
  - Repository: brand-to-eg-profile-sync
  - PR Number: 596


### Step 2: Connect to GitHub and Fetch PR Diff

This cell connects to GitHub using the PyGithub library and retrieves the diff for the specified PR.

In [17]:
from github import Github
import requests

# Initialize GitHub client
g = Github(GITHUB_TOKEN)

# Get the repository
repo = g.get_repo(f"{REPO_OWNER}/{REPO_NAME}")
print(f"Connected to repository: {repo.full_name}")

# Get the pull request
pr = repo.get_pull(PR_NUMBER)
print(f"\nPR Title: {pr.title}")
print(f"PR State: {pr.state}")
print(f"Files Changed: {pr.changed_files}")

# Fetch the diff
# GitHub API provides diff in unified diff format
headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3.diff"
}
diff_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/pulls/{PR_NUMBER}"
response = requests.get(diff_url, headers=headers)

if response.status_code == 200:
    pr_diff = response.text
    print(f"\n✓ Successfully fetched PR diff ({len(pr_diff)} characters)")
    
    # Show a preview of the diff
    print("\nDiff Preview (first 500 characters):")
    print("-" * 80)
    print(pr_diff[:500])
    print("-" * 80)
else:
    print(f"✗ Error fetching diff: {response.status_code}")
    pr_diff = None

/var/folders/r_/khgdwy2d207441v3pvjlbv9m0000gn/T/ipykernel_68287/2210705819.py:5: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g = Github(GITHUB_TOKEN)


Connected to repository: eg-internal/brand-to-eg-profile-sync

PR Title: fix(deps): update dependency ch.qos.logback:logback-core to v1.5.19 [security]
PR State: open
Files Changed: 1

PR Title: fix(deps): update dependency ch.qos.logback:logback-core to v1.5.19 [security]
PR State: open
Files Changed: 1

✓ Successfully fetched PR diff (407 characters)

Diff Preview (first 500 characters):
--------------------------------------------------------------------------------
diff --git a/pom.xml b/pom.xml
index 6afb3fa8..f708cae6 100644
--- a/pom.xml
+++ b/pom.xml
@@ -140,7 +140,7 @@
         <dependency>
             <groupId>ch.qos.logback</groupId>
             <artifactId>logback-core</artifactId>
-            <version>1.5.13</version>
+            <version>1.5.19</version>
         </dependency>
         <dependency>
             <groupId>net.logstash.logback</groupId>

--------------------------------------------------------------------------------

✓ Successfully fetched PR diff (407 

### Step 3: Extract POM Changes and Analyze with Ollama

This cell filters the diff for POM file changes and uses Ollama to extract library updates as a JSON object.

In [18]:
import ollama
import re
import json

# Filter diff for POM files only
pom_diff_sections = []
current_file = None
current_diff = []

for line in pr_diff.split('\n'):
    if line.startswith('diff --git'):
        # Save previous file's diff if it was a POM
        if current_file and 'pom.xml' in current_file:
            pom_diff_sections.append({
                'file': current_file,
                'diff': '\n'.join(current_diff)
            })
        # Start new file
        current_file = line
        current_diff = [line]
    else:
        current_diff.append(line)

# Don't forget the last file
if current_file and 'pom.xml' in current_file:
    pom_diff_sections.append({
        'file': current_file,
        'diff': '\n'.join(current_diff)
    })

print(f"Found {len(pom_diff_sections)} POM file(s) with changes\n")

# Prepare prompt for Ollama
if pom_diff_sections:
    # Combine all POM diffs
    combined_pom_diff = "\n\n".join([f"File: {section['file']}\n{section['diff']}" 
                                      for section in pom_diff_sections])
    
    prompt = f"""Analyze the following POM file diff(s) and extract all library/dependency version updates.

Return ONLY a valid JSON object with this structure:
{{
  "library_updates": [
    {{
      "library": "groupId:artifactId",
      "from_version": "X.Y.Z",
      "to_version": "A.B.C"
    }}
  ]
}}

If there are no version updates, return:
{{
  "library_updates": []
}}

Do not include any explanation or markdown formatting, only the JSON object.

Here is the diff:

{combined_pom_diff}
"""
    
    print("Sending request to Ollama...")
    print("=" * 80)
    
    # Call Ollama
    response = ollama.chat(
        model='llama3',  # Change to your preferred model
        messages=[{
            'role': 'user',
            'content': prompt
        }],
        format='json'  # Request JSON format
    )
    
    ollama_response = response['message']['content']
    print("Ollama Analysis Complete!")
    print("=" * 80)
    print("\nRaw JSON Response:")
    print(ollama_response)
else:
    print("No POM files found in the PR diff.")

Found 1 POM file(s) with changes

Sending request to Ollama...
Ollama Analysis Complete!

Raw JSON Response:
{
  "library_updates": [
    {
      "library": "ch.qos.logback:logback-core",
      "from_version": "1.5.13",
      "to_version": "1.5.19"
    }
  ]
}
Ollama Analysis Complete!

Raw JSON Response:
{
  "library_updates": [
    {
      "library": "ch.qos.logback:logback-core",
      "from_version": "1.5.13",
      "to_version": "1.5.19"
    }
  ]
}
